# Silver

## Configuración

In [ ]:
from pathlib import Path
import duckdb

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
con = duckdb.connect(str(root / 'data/processed/database.duckdb'))
con.execute('LOAD spatial')
con.execute('CREATE SCHEMA IF NOT EXISTS silver')

## Zonas y agregaciones territoriales

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE silver.zona AS
WITH provincias AS (
    SELECT DISTINCT distrito_mitma, CASE WHEN regexp_full_match(distrito_mitma, '34[0-9]{9}')
        THEN substr(distrito_mitma, 5, 2) ELSE left(municipio_ine, 2) END AS provincia_id
    FROM bronze.mitma_relacion_ine
)
SELECT d.ID AS id, d.name AS nombre,
    ST_SetCRS(ST_GeomFromText(p.geom_wkt), 'EPSG:3042') AS poligono,
    ST_SetCRS(ST_GeomFromText(c.geom_wkt), 'EPSG:3042') AS centroide,
    t.provincia_id, pr.PROVINCIA AS provincia_nombre, pr.CODAUTO AS comunidad_autonoma_id,
    pr.COMUNIDAD_AUTONOMA AS comunidad_autonoma_nombre, ST_Area(poligono)/1e6 AS superficie_km2
FROM bronze.mitma_nombres_zonas d
LEFT JOIN bronze.mitma_zonificacion_distritos p USING (ID)
LEFT JOIN bronze.mitma_centroides_distritos c USING (ID)
LEFT JOIN provincias t ON t.distrito_mitma = d.ID
LEFT JOIN bronze.ine_provincias_ccaa pr ON pr.CPRO = t.provincia_id
WHERE d.tipo_zona = 'distrito';
CREATE OR REPLACE TABLE silver.agregaciones_zona AS
SELECT DISTINCT distrito_mitma AS id_zona, municipio_mitma AS municipio, gau_mitma AS gau
FROM bronze.mitma_relacion_ine
""")

## Festivos y ámbito territorial

In [ ]:
from datetime import date

MONTHS = 'ENERO FEBRERO MARZO ABRIL MAYO JUNIO JULIO AGOSTO SEPTIEMBRE OCTUBRE NOVIEMBRE DICIEMBRE'.split()
BOE_COMMUNITIES = {
    'Andalucía': '01', 'Aragón': '02', 'Asturias': '03', 'Illes Balears': '04',
    'Canarias (1)': '05', 'Cantabria': '06', 'Castilla y León': '07', 'Castilla-La Mancha': '08',
    'Cataluña (2)': '09', 'Comunitat Valenciana': '10', 'Extremadura': '11', 'Galicia': '12',
    'Com. Madrid': '13', 'Región Murcia': '14', 'C. Foral Navarra': '15', 'País Vasco': '16',
    'La Rioja': '17', 'Ciudad de Ceuta (3)': '18', 'Ciudad de Melilla': '19',
}

entries = []
result = con.execute('SELECT * FROM bronze.boe_calendario_festivos_2023 ORDER BY rowid')
columns = [col[0] for col in result.description]
for values in result.fetchall():
    row = dict(zip(columns, values))
    label = row['Fecha de las fiestas'].strip()
    if label in MONTHS:
        month = MONTHS.index(label) + 1
        continue
    day, _, name = label.partition(' ')
    for heading, community in BOE_COMMUNITIES.items():
        if row[heading] and row[heading].strip():
            entries.append((date(2023, month, int(day)), name.strip().rstrip('.'), community))
con.execute('CREATE OR REPLACE TEMP TABLE ambitos(fecha DATE, nombre_festivo VARCHAR, comunidad VARCHAR)')
con.executemany('INSERT INTO ambitos VALUES (?, ?, ?)', entries)
con.execute("""
CREATE OR REPLACE TABLE silver.festivos AS
SELECT row_number() OVER (ORDER BY fecha, nombre_festivo)::INTEGER AS id_festivo, *
FROM (SELECT DISTINCT fecha, nombre_festivo FROM ambitos);
CREATE OR REPLACE TABLE silver.relacion_festivos AS
SELECT DISTINCT f.id_festivo, z.id AS id_zona FROM ambitos a
JOIN silver.festivos f USING (fecha, nombre_festivo)
JOIN silver.zona z ON z.comunidad_autonoma_id = a.comunidad
""")

## Viajes y pernoctaciones

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE silver.viajes AS
SELECT strptime(fecha || lpad(periodo, 2, '0'), '%Y%m%d%H') AS fecha,
    origen AS id_zona_origen, destino AS id_zona_destino, sum(viajes::DOUBLE) AS viajes
FROM bronze.mitma_viajes WHERE fecha BETWEEN '20220101' AND '20241231' GROUP BY 1,2,3;
CREATE OR REPLACE TABLE silver.mitma_pernoctaciones AS
SELECT strptime(fecha, '%Y%m%d') AS fecha, zona_residencia AS id_zona_residencia,
    zona_pernoctacion AS id_zona_pernoctacion, personas::DOUBLE AS personas
FROM bronze.mitma_pernoctaciones WHERE fecha BETWEEN '20220101' AND '20241231'
""")

## Correspondencia INE–MITMA y pesos de población

In [ ]:
con.execute("""
CREATE OR REPLACE TEMP VIEW mapa AS
SELECT DISTINCT seccion_ine, distrito_mitma AS id_zona, municipio_ine FROM bronze.mitma_relacion_ine;
CREATE OR REPLACE TEMP MACRO seccion(s) AS regexp_extract(s, '^([0-9]{10})( |$)', 1);
CREATE OR REPLACE TEMP MACRO anual(p) AS make_date(try_cast(p AS INTEGER), 1, 1);
CREATE OR REPLACE TEMP MACRO numero(v) AS try_cast(replace(replace(trim(v), '.', ''), ',', '.') AS DOUBLE);
CREATE OR REPLACE TEMP VIEW pesos AS
SELECT seccion(Secciones) AS seccion_ine, anual(Periodo) AS año, numero(Total) AS personas
FROM bronze.ine_poblacion WHERE Periodo BETWEEN '2022' AND '2024' AND Secciones IS NOT NULL AND Sexo = 'Total' AND "Lugar de nacimiento" = 'Total'
""")

## poblacion

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE silver.poblacion AS
SELECT m.id_zona, anual(Periodo) AS año, "Sexo" AS sexo, "Lugar de nacimiento" AS pais_nacimiento,
    sum(numero(Total))::INTEGER AS total
FROM bronze.ine_poblacion s JOIN mapa m ON m.seccion_ine = seccion(s.Secciones)
WHERE "Sexo" <> 'Total' AND "Lugar de nacimiento" NOT IN ('Total', 'Total CNAE') AND Periodo BETWEEN '2022' AND '2024'
GROUP BY 1,2,3,4
""")

## situacion_laboral

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE silver.situacion_laboral AS
SELECT m.id_zona, anual(Periodo) AS año, "País de nacimiento" AS pais_nacimiento, "Relación con la actividad" AS situacion,
    sum(numero(Total))::INTEGER AS total
FROM bronze.ine_relacion_actividad s JOIN mapa m ON m.seccion_ine = seccion(s.Secciones)
WHERE "País de nacimiento" <> 'Total' AND "Relación con la actividad" NOT IN ('Total', 'Total CNAE') AND Periodo BETWEEN '2022' AND '2024'
GROUP BY 1,2,3,4
""")

## nivel_estudios

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE silver.nivel_estudios AS
SELECT m.id_zona, anual(Periodo) AS año, "País de nacimiento" AS pais_nacimiento, "Nivel de formación alcanzado" AS nivel_estudios,
    sum(numero(Total))::INTEGER AS total
FROM bronze.ine_nivel_estudios s JOIN mapa m ON m.seccion_ine = seccion(s.Secciones)
WHERE "País de nacimiento" <> 'Total' AND "Nivel de formación alcanzado" NOT IN ('Total', 'Total CNAE') AND Periodo BETWEEN '2022' AND '2024'
GROUP BY 1,2,3,4
""")

## rama_actividad

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE silver.rama_actividad AS
SELECT m.id_zona, anual(Periodo) AS año, "Sexo" AS sexo, "Actividad económica" AS rama_actividad,
    sum(numero(Total))::INTEGER AS total
FROM bronze.ine_rama_actividad s JOIN mapa m ON m.seccion_ine = seccion(s.Secciones)
WHERE "Sexo" <> 'Total' AND "Actividad económica" NOT IN ('Total', 'Total CNAE') AND Periodo BETWEEN '2022' AND '2024'
GROUP BY 1,2,3,4
""")

## Renta

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE silver.renta AS
WITH indicadores AS (
    SELECT m.id_zona, anual(r.Periodo) AS año, r."Indicadores de renta media" AS indicador,
           weighted_avg(numero(r.Total), p.personas) AS valor
    FROM bronze.ine_renta r JOIN mapa m ON m.seccion_ine = seccion(r.Secciones)
    LEFT JOIN pesos p ON p.seccion_ine = m.seccion_ine AND p.año = anual(r.Periodo) AND p.personas > 0
    WHERE r.Periodo BETWEEN '2022' AND '2024' AND r."Indicadores de renta media" IN ('Renta bruta media por persona','Renta neta media por persona',
        'Renta bruta media por hogar','Renta neta media por hogar') GROUP BY 1,2,3
)
SELECT id_zona, año,
    max(valor) FILTER (WHERE indicador = 'Renta bruta media por persona') AS renta_media_bruta_persona,

    max(valor) FILTER (WHERE indicador = 'Renta bruta media por hogar') AS renta_media_bruta_hogar,

    max(valor) FILTER (WHERE indicador = 'Renta neta media por persona') AS renta_media_neta_persona,

    max(valor) FILTER (WHERE indicador = 'Renta neta media por hogar') AS renta_media_neta_hogar
FROM indicadores GROUP BY 1,2
""")

## Vehículos

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE silver.vehiculos AS
WITH d AS (
    SELECT lpad("Código INE",5,'0') AS municipio_ine,
        anual(regexp_extract(_archivo, '([0-9]{4})[.]xlsx$', 1)) AS año, *
    FROM bronze.dgt_datos_municipales
), territorios AS (
    SELECT DISTINCT municipio_ine, id_zona FROM mapa
), unicos AS (
    SELECT municipio_ine, min(id_zona) AS id_zona FROM territorios GROUP BY 1 HAVING count(*) = 1
), fragmentos AS (
    SELECT m.municipio_ine, p.año, m.id_zona, sum(p.personas) AS personas, count(p.personas) AS secciones
    FROM mapa m JOIN pesos p USING (seccion_ine) ANTI JOIN unicos u USING (municipio_ine)
    GROUP BY 1,2,3
), repartos AS (
    SELECT f.*, sum(personas) OVER w AS poblacion, sum(secciones) OVER w AS disponibles
    FROM fragmentos f WINDOW w AS (PARTITION BY municipio_ine, año)
), cuotas AS (
    SELECT u.municipio_ine, u.id_zona, y.año, 1.0 AS peso FROM unicos u CROSS JOIN (SELECT DISTINCT año FROM d) y
    UNION ALL
    SELECT municipio_ine, id_zona, año, personas / poblacion FROM repartos
    JOIN (SELECT municipio_ine, count(*) AS n FROM mapa GROUP BY 1) USING (municipio_ine)
    WHERE disponibles = n AND poblacion > 0
), medidas AS (
    UNPIVOT d ON "Censo Conductores", "Parque Ciclomotores", "Parque Motocicletas", "Parque Turismos", "Parque Furgonetas", "Parque Camiones", "Parque Total" INTO NAME medida VALUE valor
), partes AS (
    SELECT municipio_ine, año, id_zona, medida, valor::INTEGER AS total, total*peso AS cuota
    FROM medidas JOIN cuotas USING (municipio_ine, año)
), redondeo AS (
    SELECT *, total - sum(floor(cuota)) OVER w AS sobrante,
        row_number() OVER (w ORDER BY cuota-floor(cuota) DESC, id_zona) AS turno
    FROM partes WINDOW w AS (PARTITION BY municipio_ine, año, medida)
)
SELECT año, id_zona,
    sum(floor(cuota)+(turno<=sobrante)::INT) FILTER (WHERE medida='Censo Conductores')::INTEGER AS censo_conductores,

    sum(floor(cuota)+(turno<=sobrante)::INT) FILTER (WHERE medida='Parque Ciclomotores')::INTEGER AS parque_ciclomotores,

    sum(floor(cuota)+(turno<=sobrante)::INT) FILTER (WHERE medida='Parque Motocicletas')::INTEGER AS parque_motocicletas,

    sum(floor(cuota)+(turno<=sobrante)::INT) FILTER (WHERE medida='Parque Turismos')::INTEGER AS parque_turismos,

    sum(floor(cuota)+(turno<=sobrante)::INT) FILTER (WHERE medida='Parque Furgonetas')::INTEGER AS parque_furgonetas,

    sum(floor(cuota)+(turno<=sobrante)::INT) FILTER (WHERE medida='Parque Camiones')::INTEGER AS parque_camiones,

    sum(floor(cuota)+(turno<=sobrante)::INT) FILTER (WHERE medida='Parque Total')::INTEGER AS parque_total
FROM redondeo GROUP BY 1,2
""")

## Meteorología

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE silver.meteorologia AS
SELECT id_distrito AS id_zona, hora_utc::TIMESTAMP AS fecha,
    "2m_temperature"::DOUBLE - 273.15 AS temperatura,
    CASE WHEN total_precipitation::DOUBLE < 0 THEN 0.0 ELSE total_precipitation::DOUBLE * 1000 END AS precipitaciones, snowfall::DOUBLE * 1000 AS nieve,
    sqrt("10m_u_component_of_wind"::DOUBLE^2 + "10m_v_component_of_wind"::DOUBLE^2) AS velocidad_viento
FROM bronze.copernicus_horario WHERE hora_utc >= '2022' AND hora_utc < '2025'
""")

## Exportación a CSV

In [ ]:
destino = root / 'data/silver'
destino.mkdir(parents=True, exist_ok=True)
tablas = ['zona', 'agregaciones_zona', 'festivos', 'relacion_festivos', 'viajes', 'mitma_pernoctaciones',
          'poblacion', 'situacion_laboral', 'rama_actividad', 'nivel_estudios', 'renta', 'vehiculos', 'meteorologia']
for tabla in tablas:
    campos = ('* REPLACE (ST_AsText(poligono) AS poligono, ST_AsText(centroide) AS centroide)'
              if tabla == 'zona' else '*')
    con.execute(f'COPY (SELECT {campos} FROM silver.{tabla}) TO ? (FORMAT CSV, HEADER)',
                [str(destino / f'{tabla}.csv')])
con.close()